# Synthetic Time Series Data Generation

Generates 1,000 distinct monthly time series, each exactly 5 years (60 months) long, and saves them to `<catalog>.<schema>.train`.

## 1. Parameters

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "mmf"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "synthetic"

table_name = f"{catalog}.{schema}.train"
print(f"Target table: {table_name}")

## 2. Setup — Create Catalog and Schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

## 3. Generate Synthetic Time Series

In [ ]:
import numpy as np
import pandas as pd
from datetime import date

np.random.seed(42)

N_SERIES = 1000
N_YEARS = 5
N_MONTHS = N_YEARS * 12  # 60 months per series
END_DATE = date(2024, 12, 1)

dates = pd.date_range(end=END_DATE, periods=N_MONTHS, freq="MS")

records = []

for i in range(N_SERIES):
    unique_id = f"series_{i:04d}"

    trend_slope = np.random.uniform(0.5, 5.0)
    level = np.random.uniform(50, 500)
    seasonality_amp = np.random.uniform(5, 50)
    noise_std = np.random.uniform(1, 20)

    t = np.arange(N_MONTHS)
    seasonal = seasonality_amp * np.sin(2 * np.pi * t / 12)
    noise = np.random.normal(0, noise_std, N_MONTHS)
    y = level + trend_slope * t + seasonal + noise
    y = np.clip(y, 0, None)

    for dt, val in zip(dates, y):
        records.append((unique_id, dt.date(), float(val)))

pdf = pd.DataFrame(records, columns=["unique_id", "ds", "y"])
print(f"Total rows: {len(pdf):,}")
print(f"Series count: {pdf['unique_id'].nunique()}")
print(f"Months per series: {N_MONTHS}")
print(f"Date range: {pdf['ds'].min()} → {pdf['ds'].max()}")
pdf.head()

## 3b. Inject Anomalies and Missing Entries

Contaminates ~10% of series with realistic data-quality issues:
- **Spikes** — sudden values 5–10× the local mean
- **Drops** — values forced to zero or near-zero
- **Nulls** — `y` set to `NaN` (sensor dropout / ETL failure)
- **Gaps** — entire rows removed (missing months)

In [ ]:
rng = np.random.RandomState(99)

all_series = pdf["unique_id"].unique()
n_affected = int(len(all_series) * 0.10)
affected_series = rng.choice(all_series, size=n_affected, replace=False)

# Split affected series into four non-overlapping groups
rng.shuffle(affected_series)
spike_ids = affected_series[: n_affected // 4]
drop_ids  = affected_series[n_affected // 4 : n_affected // 2]
null_ids  = affected_series[n_affected // 2 : 3 * n_affected // 4]
gap_ids   = affected_series[3 * n_affected // 4 :]

# --- Spikes: multiply 2–4 random points by 5–10× ---
for sid in spike_ids:
    mask = pdf["unique_id"] == sid
    idx = pdf.index[mask]
    n_spikes = rng.randint(2, 5)
    spike_idx = rng.choice(idx, size=min(n_spikes, len(idx)), replace=False)
    pdf.loc[spike_idx, "y"] *= rng.uniform(5, 10, size=len(spike_idx))

# --- Drops: force 2–4 random points to zero or near-zero ---
for sid in drop_ids:
    mask = pdf["unique_id"] == sid
    idx = pdf.index[mask]
    n_drops = rng.randint(2, 5)
    drop_idx = rng.choice(idx, size=min(n_drops, len(idx)), replace=False)
    pdf.loc[drop_idx, "y"] = rng.uniform(0, 1, size=len(drop_idx))

# --- Nulls: set 3–6 random y values to NaN ---
for sid in null_ids:
    mask = pdf["unique_id"] == sid
    idx = pdf.index[mask]
    n_nulls = rng.randint(3, 7)
    null_idx = rng.choice(idx, size=min(n_nulls, len(idx)), replace=False)
    pdf.loc[null_idx, "y"] = np.nan

# --- Gaps: remove 3–6 consecutive months ---
gap_rows_to_drop = []
for sid in gap_ids:
    mask = pdf["unique_id"] == sid
    idx = pdf.index[mask].tolist()
    gap_len = rng.randint(3, 7)
    if len(idx) > gap_len + 2:
        start = rng.randint(1, len(idx) - gap_len - 1)
        gap_rows_to_drop.extend(idx[start : start + gap_len])

pdf = pdf.drop(index=gap_rows_to_drop).reset_index(drop=True)

n_spikes_total = len(spike_ids)
n_drops_total = len(drop_ids)
n_nulls_total = int(pdf["y"].isna().sum())
n_gaps_total = len(gap_rows_to_drop)

print(f"Injected anomalies into {n_affected} / {len(all_series)} series:")
print(f"  Spike series : {n_spikes_total}")
print(f"  Drop series  : {n_drops_total}")
print(f"  Null values  : {n_nulls_total}")
print(f"  Removed rows : {n_gaps_total} (gap series: {len(gap_ids)})")
print(f"Final row count: {len(pdf):,}")

## 4. Save to Delta Table

In [ ]:
sdf = spark.createDataFrame(pdf)

(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(f"Table written: {table_name}")

## 5. Verify

In [ ]:
result = spark.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT unique_id)         AS n_series,
        MIN(ds)                           AS min_date,
        MAX(ds)                           AS max_date,
        ROUND(AVG(y), 2)                  AS avg_y,
        SUM(CASE WHEN y IS NULL THEN 1 ELSE 0 END) AS null_count,
        ROUND(MAX(y), 2)                  AS max_y,
        ROUND(MIN(y), 2)                  AS min_y
    FROM {table_name}
""").toPandas()

display(result)

In [ ]:
# Verify all series have equal length (expect 60 months each, except gap-affected series)
length_dist = spark.sql(f"""
    SELECT n_months, COUNT(*) AS n_series
    FROM (
        SELECT unique_id, COUNT(*) AS n_months
        FROM {table_name}
        GROUP BY unique_id
    )
    GROUP BY n_months
    ORDER BY n_months
""")
length_dist.show()